In [ ]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import time
import re

# 1. Load the initial dataset and extract the names
print("Loading initial dataset...")
df = pd.read_csv('../q/test_scrape.csv')
games_list = df['game'].tolist()

new_dataset = []
print(f"Starting extraction for {len(games_list)} games...\n")

# 2. Define headers to mimic a web browser (Bypasses 401 Unauthorized errors)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

# 3. Loop through each game
for original_name in games_list:
    # CLEANING: Remove Wikipedia tags like " (role-playing game)"
    search_name = re.sub(r' \(.*?\)', '', str(original_name)).strip()
    print(f"Processing: {search_name}...")
    
    try:
        # --- PHASE 1: Search API (Get the ID) ---
        # We use boardgamegeek.com instead of rpggeek.com to avoid Cloudflare blocks. 
        # The parameter type=rpgitem ensures we only search for RPGs.
        search_url = "https://boardgamegeek.com/xmlapi2/search"
        search_params = {"query": search_name, "type": "rpgitem", "exact": 1}
        
        search_res = requests.get(search_url, params=search_params, headers=headers)
        time.sleep(2) # MANDATORY sleep to avoid IP bans (429 Too Many Requests)
        
        if search_res.status_code == 200:
            root = ET.fromstring(search_res.content)
            item = root.find('item')
            
            if item is not None:
                game_id = item.get('id')
                
                # --- PHASE 2: Thing API (Get the Stats and Details) ---
                thing_url = "https://boardgamegeek.com/xmlapi2/thing"
                thing_params = {"id": game_id, "stats": 1}
                
                thing_res = requests.get(thing_url, params=thing_params, headers=headers)
                time.sleep(2) # MANDATORY sleep
                
                if thing_res.status_code == 200:
                    t_root = ET.fromstring(thing_res.content)
                    t_item = t_root.find('item')
                    
                    if t_item is not None:
                        # Extract Description
                        desc_tag = t_item.find('description')
                        description = desc_tag.text if desc_tag is not None else "No description"
                        
                        # Extract Average Score
                        average_score = None
                        avg_tag = t_item.find('.//statistics/ratings/average')
                        if avg_tag is not None:
                            average_score = avg_tag.get('value')
                            
                        # Extract Number of Reviews/Ratings
                        num_ratings = None
                        ratings_tag = t_item.find('.//statistics/ratings/usersrated')
                        if ratings_tag is not None:
                            num_ratings = ratings_tag.get('value')
                            
                        # Save the successfully scraped data
                        new_dataset.append({
                            'Name': search_name,
                            'Description': description,
                            'Average Score': average_score,
                            'Number of Reviews': num_ratings
                        })
                        print(f"  -> Success! Score: {average_score} | Reviews: {num_ratings}")
                        continue 
                    else:
                        print("  -> Game details not found in Thing API.")
                else:
                    print(f"  -> Thing API failed. Status Code: {thing_res.status_code}")
            else:
                print("  -> Game not found in Search API results.")
        else:
            print(f"  -> Search API failed. Status Code: {search_res.status_code}")
            
        # If any step above fails normally (e.g. not found), append empty values so we don't lose the row
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})
            
    except Exception as e:
        print(f"  -> Error processing {search_name}: {e}")
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})

Loading initial dataset...
Starting extraction for 775 games...

Processing: 13th Age...
  -> Search API failed. Status Code: 401
Processing: The 23rd Letter...
  -> Search API failed. Status Code: 401
Processing: 2300 AD...


KeyboardInterrupt: 

In [ ]:
# 4. Save to a new CSV file
new_df = pd.DataFrame(new_dataset)
new_df.to_csv('ttrpg_api_dataset.csv', index=False)
print("\nExtraction Complete! Data saved to 'ttrpg_api_dataset.csv'")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
import time

print("Loading initial dataset...")
try:
    df = pd.read_csv('test_scrape.csv')
    games_list = df['game'].tolist()
except FileNotFoundError:
    df = pd.read_csv('../q/test_scrape.csv')
    games_list = df['game'].tolist()

new_dataset = []
print(f"Starting Web Scrape for {len(games_list)} games...\n")

# Headers are critical for scraping to avoid being blocked as a bot
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

for original_name in games_list:
    # CLEANING: Remove Wikipedia tags
    search_name = re.sub(r' \(.*?\)', '', str(original_name)).strip()
    print(f"Scraping: {search_name}...")
    
    try:
        # --- PHASE 1: Scrape the Search Page to get the Game's URL ---
        # We use the standard web search URL, not the API
        search_url = "https://boardgamegeek.com/geeksearch.php"
        search_params = {"action": "search", "objecttype": "rpgitem", "q": search_name}
        
        search_res = requests.get(search_url, params=search_params, headers=headers)
        time.sleep(2) # Respect the server
        
        if search_res.status_code == 200:
            soup = BeautifulSoup(search_res.content, 'html.parser')
            
            # Find the first link in the search results that points to an RPG item
            link_tag = soup.find('a', href=re.compile(r'^/rpgitem/'))
            
            if link_tag:
                # Construct the full URL to the game's actual webpage
                item_url = "https://boardgamegeek.com" + link_tag['href']
                
                # --- PHASE 2: Scrape the Game's Webpage ---
                item_res = requests.get(item_url, headers=headers)
                time.sleep(5) 
                
                if item_res.status_code == 200:
                    item_soup = BeautifulSoup(item_res.content, 'html.parser')
                    
                    # 1. Extract Description from the SEO Meta Tags (Easiest HTML scraping)
                    meta_desc = item_soup.find('meta', attrs={'name': 'description'})
                    description = meta_desc['content'] if meta_desc else "No description"
                    
                    # 2. Extract Stats using BeautifulSoup + RegEx!
                    # BGG hides the score in a Javascript variable called GEEK.geekitemPreload
                    average_score = None
                    num_ratings = None
                    
                    # Find all script tags on the page
                    scripts = item_soup.find_all('script')
                    for script in scripts:
                        if script.string and 'GEEK.geekitemPreload' in script.string:
                            # Use RegEx to extract the JSON dictionary hidden inside the Javascript text
                            json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                            
                            if json_match:
                                # Convert the extracted string into a Python dictionary
                                hidden_data = json.loads(json_match.group(1))
                                
                                # Navigate the dictionary to find the stats
                                stats = hidden_data.get('item', {}).get('stats', {})
                                average_score = stats.get('average')
                                num_ratings = stats.get('usersrated')
                            break # We found the data, stop checking other scripts
                            
                    # Save the scraped row
                    new_dataset.append({
                        'Name': search_name,
                        'Description': description,
                        'Average Score': average_score,
                        'Number of Reviews': num_ratings
                    })
                    print(f"  -> Scrape Success! Score: {average_score} | Reviews: {num_ratings}")
                    continue
                else:
                    print(f"  -> Failed to load item page. Status: {item_res.status_code}")
            else:
                print("  -> Game not found in search results.")
        else:
            print(f"  -> Search page failed. Status: {search_res.status_code}")
            
        # Append empty row if failed
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})
            
    except Exception as e:
        print(f"  -> Error scraping {search_name}: {e}")
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})



Loading initial dataset...
Starting Web Scrape for 775 games...

Scraping: 13th Age...
  -> Scrape Success! Score: 7.90114 | Reviews: 131
Scraping: The 23rd Letter...
  -> Failed to load item page. Status: 403
Scraping: 2300 AD...
  -> Failed to load item page. Status: 403
Scraping: 3D&T...
  -> Failed to load item page. Status: 403
Scraping: 7th Sea...
  -> Failed to load item page. Status: 403
Scraping: Aberrant...


KeyboardInterrupt: 

In [ ]:
# Save to CSV
new_df = pd.DataFrame(new_dataset)
new_df.to_csv('ttrpg_scraped_dataset.csv', index=False)
print("\nWeb Scrape Complete! Data saved to 'ttrpg_scraped_dataset.csv'")